# Find a chemical structure in your documents

Upload PDFs / Word files, give a SMILES, and get back **which document, which
page, and where on the page** the structure is drawn.

Pipeline (PatCID, *Nature Communications* 15:6532, 2024):
**DECIMER-Segmentation** → **MolClassifier** → **DECIMER / MolGrapher** → **RDKit matching**

Models and the per-document extraction cache are kept on your Google Drive, so
the second time you open this notebook nothing is re-downloaded and documents
you already processed are searched instantly.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Everything
works on CPU too, just several times slower (a 50-page patent is minutes on GPU,
tens of minutes on CPU).

## 1. Check the runtime

In [ ]:
#@title Runtime check — the Python version decides which engines you can use
import sys, subprocess

print("Python:", sys.version.split()[0])
try:
    print(subprocess.check_output(["nvidia-smi", "-L"], text=True).strip())
except Exception:
    print("GPU: none (CPU-only run — this works, just slower)")

if sys.version_info[:2] != (3, 11):
    print(
        "\nNOTE: MolGrapher pins its torch wheels to CPython 3.11, so on this\n"
        "runtime use the DECIMER recognition engine (section 2). It is the\n"
        "engine PatCID benchmarked at 67.2% vs MolGrapher's 63.0% on D2C-RND,\n"
        "so you lose nothing on accuracy — only some CPU speed."
    )

## 2. Install

> **If you have run any earlier version of this notebook, do this first:**
> **Runtime → Disconnect and delete runtime**, then re-run from the top.
>
> Not just "Restart session" — delete. An earlier revision of this
> notebook downgraded numpy and compiled it from source, and TensorFlow
> reads `TF_USE_LEGACY_KERAS` only at its first import. A deleted runtime
> clears both; a restart clears only the second.

Two things here are not plain `pip install`s, both because
`decimer-segmentation` predates the current TensorFlow/numpy world.

**1. `--no-deps`.** It declares `tensorflow>=2.12,<=2.15.1`, and TensorFlow
ships no wheel in that range for Colab's Python (3.13 — cp313 wheels start at
TF 2.20), so a normal install ends in `ResolutionImpossible`. That cap is a
proxy for **numpy < 2** (the package touches `np.VisibleDeprecationWarning`,
which numpy 2.0 removed) — but numpy only supports Python 3.13 from 2.1, so
"numpy < 2" is not installable here either. `structure_finder` restores that
one attribute instead; it is the package's only numpy-2 incompatibility.

**2. `tf-keras`.** Loading the Mask R-CNN `.h5` goes through
`hdf5_format.load_weights_from_hdf5_group_by_name`, which requires every layer
weight to be a `tf.Variable`. Keras 3 (TensorFlow ≥ 2.16) uses `keras.Variable`,
so the load fails with *"Save or restore weights that is not an instance of
`tf.Variable` is not supported in h5"*. Setting `TF_USE_LEGACY_KERAS=1` with
`tf-keras` installed points `tf.keras` back at Keras 2, where it works —
verified on TensorFlow 2.19.1 and 2.20.0.

`tf-keras` must match TensorFlow's **major.minor**. A mismatched one is
imported, rejected, and silently replaced by Keras 3 — failing exactly as
if it were not installed at all. The cell below reads the installed
TensorFlow version and pins to match, so do not replace it with a plain
`pip install tf-keras`.

`structure_finder` sets that variable when you import it. TensorFlow reads it
at **its** import time, so import `structure_finder` first — the cells below
do. No numpy pin.

In [ ]:
#@title Install (~5 min on a fresh runtime)
# decimer-segmentation, minus its unsatisfiable tensorflow<=2.15.1 cap ...
!pip install -q --no-deps decimer-segmentation
# ... plus the dependencies that cap would have brought in
!pip install -q scikit-image opencv-python matplotlib IPython PyMuPDF numba scipy requests

# Recognition. Its own cap (tensorflow<=2.20.0) IS satisfiable on Python 3.13.
!pip install -q decimer

# Keras 2. DECIMER-Segmentation loads its Mask R-CNN .h5 through
# hdf5_format.load_weights_from_hdf5_group_by_name, which requires every layer
# weight to be a tf.Variable. Under Keras 3 (TensorFlow >= 2.16) they are
# keras.Variable, and loading dies with "Save or restore weights that is not an
# instance of `tf.Variable` is not supported in h5". tf-keras +
# TF_USE_LEGACY_KERAS=1 points tf.keras back at Keras 2, where it works.
# tf-keras MUST match TensorFlow's major.minor. If it does not, TensorFlow
# imports it, rejects it, and silently falls back to Keras 3 - the weight load
# then fails exactly as if tf-keras were absent. Read the installed TensorFlow
# version from package metadata (which does not import it, so TF_USE_LEGACY_KERAS
# is still free to take effect later) and pin to match.
import subprocess, sys
from importlib.metadata import PackageNotFoundError, version

try:
    _tf_version = version("tensorflow")
    _spec = "tf-keras~=" + ".".join(_tf_version.split(".")[:2]) + ".0"
except PackageNotFoundError:
    _tf_version, _spec = "not installed", "tf-keras"

print(f"TensorFlow {_tf_version} -> installing {_spec}")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", _spec], check=True)

# Core: matching, PDF/DOCX handling, reporting
!pip install -q rdkit pymupdf python-docx

# .docx -> PDF, so Word documents keep their real page numbers
!apt-get -qq install -y libreoffice-writer > /dev/null

print("\nInstall complete.")

# TF_USE_LEGACY_KERAS is only read when TensorFlow is FIRST imported. If this
# session already imported it, no amount of setting the variable now will help -
# the session has to restart. Do it automatically rather than printing advice
# that is easy to scroll past.
import sys

if "tensorflow" in sys.modules:
    print("TensorFlow was already imported in this session, so Keras 2 cannot")
    print("take effect. Restarting the session now - this is expected, not a")
    print("crash. When it comes back, continue from the NEXT cell.")
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)
else:
    print("Continue to the next cell.")

### Get the tool itself

`structure_finder` lives on a branch of this fork, not on upstream `DS4SD/PatCID`.
The cell below deletes any existing `/content/PatCID` before cloning — `git clone`
fails into a non-empty directory, and a stale clone from an earlier run is the
usual reason `import structure_finder` fails with `ModuleNotFoundError`.

In [ ]:
#@title Clone structure_finder and register the import paths
import importlib, os, shutil, subprocess, sys

REPO   = "https://github.com/ranjitranbhor/PatCID.git"
BRANCH = "claude/structure-identification-documents-0kwz8a"
DEST   = "/content/PatCID"

# git clone refuses to write into a non-empty directory, and `!git clone -q`
# would report that only on stderr. Remove first, and use check=True so a
# failure raises here instead of surfacing as ModuleNotFoundError later.
if os.path.isdir(DEST):
    print(f"removing existing {DEST}")
    shutil.rmtree(DEST)

subprocess.run(["git", "clone", "-q", "--branch", BRANCH, REPO, DEST], check=True)
assert os.path.isfile(os.path.join(DEST, "structure_finder", "__init__.py")), (
    f"{DEST} was cloned but has no structure_finder package — wrong repo or branch?"
)

paths = [DEST]
if os.path.isdir("/content/MolClassifier"):
    # MolClassifier imports `albumentations_transforms` as a TOP-LEVEL module
    # even though the file lives inside the package, so its inner directory has
    # to be on the path as well as the repo root.
    paths += ["/content/MolClassifier", "/content/MolClassifier/mol_classifier"]

for path in paths:
    if path not in sys.path:
        sys.path.insert(0, path)

# This directory was very likely already on sys.path earlier in this kernel,
# back when it held a different checkout. Python caches a per-directory finder,
# and a half-imported package can linger in sys.modules. Clear both, or the
# import below can miss a package that is demonstrably on disk.
for name in [n for n in sys.modules if n.split(".")[0] == "structure_finder"]:
    del sys.modules[name]
sys.path_importer_cache.pop(DEST, None)
importlib.invalidate_caches()

try:
    import structure_finder
except ModuleNotFoundError as error:
    # Do not leave the user guessing: show what is actually on disk and on the
    # path, so the cause is visible rather than inferred.
    print("IMPORT FAILED:", error)
    print("\n--- diagnostics ---")
    print("DEST exists      :", os.path.isdir(DEST))
    print("DEST contents    :", sorted(os.listdir(DEST))[:15] if os.path.isdir(DEST) else "n/a")
    pkg = os.path.join(DEST, "structure_finder")
    print("package exists   :", os.path.isdir(pkg))
    print("package contents :", sorted(os.listdir(pkg))[:15] if os.path.isdir(pkg) else "n/a")
    print("DEST in sys.path :", DEST in sys.path)
    print("sys.path[:5]     :", sys.path[:5])
    print("\nIf the package is on disk but the import still fails, the kernel "
          "state is stale:\n  Runtime -> Restart session, then run this cell "
          "again (pip installs survive a restart).")
    raise

print("structure_finder", structure_finder.__version__)
print("import paths:", paths)

In [ ]:
#@title Verify the install (fails fast if anything is wrong)
import sys

# structure_finder sets TF_USE_LEGACY_KERAS on import, which TensorFlow reads
# when IT is first imported - so import it before tensorflow, not after.
import structure_finder
from structure_finder.compat import apply_numpy2_shims, enable_legacy_keras

print("numpy 2 shim applied", apply_numpy2_shims())
print("legacy Keras active ", enable_legacy_keras())

import numpy, tensorflow
print("python              ", sys.version.split()[0])
print("numpy               ", numpy.__version__)
print("tensorflow          ", tensorflow.__version__)
print("tf.keras            ", tensorflow.keras.__name__)

import decimer_segmentation
print("decimer-segmentation", decimer_segmentation.__version__)

# Build the Mask R-CNN AND load the published weights. Building alone is not
# enough: the weight load is the step that fails under Keras 3, so exercise the
# whole thing here rather than 20 minutes into a search.
from decimer_segmentation import get_model

get_model()
print("\nMask R-CNN built and weights loaded - segmentation is ready.")

### Optional: MolClassifier (Clean / Markush / Trash)

Recommended. It filters segmentation errors and tells you which detections are
**Markush** structures (generic structures with R groups — those can never equal
a concrete SMILES, so you want them flagged rather than silently missed).

torch and torchvision are already on Colab, so this is a small install.

In [ ]:
#@title Install MolClassifier
import os, sys, subprocess

DEST = "/content/MolClassifier"
if not os.path.isdir(DEST):
    subprocess.run(
        ["git", "clone", "-q", "https://github.com/DS4SD/MolClassifier.git", DEST],
        check=True,
    )
!pip install -q pycocotools albumentations imantics more-itertools

# MolClassifier imports `albumentations_transforms` as a TOP-LEVEL module even
# though the file lives inside the package, so its inner directory has to be on
# the path as well as the repo root. (The clone cell above ran before this one
# existed on disk, so add the paths here.)
for path in [DEST, os.path.join(DEST, "mol_classifier")]:
    if path not in sys.path:
        sys.path.insert(0, path)

from mol_classifier.classifier import ImageSeg  # noqa: F401
print("MolClassifier ready.")

### Optional: MolGrapher instead of DECIMER — **Python 3.11 runtimes only**

MolGrapher is the engine PatCID itself used (about 2× faster than DECIMER on
CPU). Its `setup.py` builds torch wheel URLs pinned to CPython 3.11, so on a
3.12+ runtime this cell **will fail** — that is expected, skip it and stay on
DECIMER. Run the cell in section 1 first to see which you have.

In [ ]:
#@title Install MolGrapher (skip unless Python is 3.11)
import sys
if sys.version_info[:2] == (3, 11):
    !git clone -q https://github.com/DS4SD/MolGrapher.git /content/MolGrapher
    !cd /content/MolGrapher && pip install -q -e ".[cpu]" && bash install_paddleocr.sh
    print("MolGrapher installed — you can set RECOGNIZER = 'molgrapher' below.")
else:
    print(f"Skipped: Python {sys.version_info.major}.{sys.version_info.minor} "
          "is not 3.11. Stay on RECOGNIZER = 'decimer'.")

In [ ]:
#@title Smoke test — the full suite, ~10 s, no model weights needed
!cd /content/PatCID && python -m pytest tests/test_structure_finder.py -q

## 3. Mount Google Drive

Everything heavy lands in `MyDrive/structure_finder/`:

```
models/        MolClassifier checkpoint
models/hf/     HF_HOME      — MolGrapher weights
models/pystow/ PYSTOW_HOME  — DECIMER weights
cache/         per-document extraction results   <- the valuable part
outputs/       reports and annotated pages
```

The cache is keyed by file hash, so a document you processed last week is never
re-processed — searching a different molecule across it takes milliseconds.

In [ ]:
#@title Mount Drive and pre-download the model weights
from structure_finder import resolve_workspace
from structure_finder.setup_models import main as fetch_weights

workspace = resolve_workspace(use_drive=True)
print("Workspace:", workspace.root, "\n")

# Called in-process, NOT as `!python -m ...`: a shell subprocess gets a fresh
# interpreter that knows nothing about the sys.path entries added above, so it
# would fail with ModuleNotFoundError.
#
# Fetching the recognition weights here too (rather than lazily on first use)
# means a download problem surfaces now instead of part-way through a search.
exit_code = fetch_weights([
    "--drive",
    "--engine", "decimer-seg",     # segmentation Mask R-CNN
    "--engine", "decimer",         # recognition (EfficientNetV2 + transformer)
    "--engine", "molclassifier",   # Clean / Markush / Trash
])

if exit_code:
    print("\nSome weights could not be fetched — see the errors above. "
          "Re-running this cell retries; anything already downloaded is skipped.")
else:
    print("\nAll model weights are on Drive.")

## 4. Upload your documents

Accepts `.pdf`, `.docx`, `.doc` and image files. You can also skip this cell and
point the search at a Drive folder instead — set `DOCUMENTS_DIR` in section 6.

In [ ]:
#@title Upload PDFs / Word documents
import os, shutil
from google.colab import files

UPLOAD_DIR = "/content/documents"
os.makedirs(UPLOAD_DIR, exist_ok=True)

uploaded = files.upload()
for name in uploaded:
    shutil.move(name, os.path.join(UPLOAD_DIR, name))

print(f"\n{len(os.listdir(UPLOAD_DIR))} document(s) in {UPLOAD_DIR}:")
for name in sorted(os.listdir(UPLOAD_DIR)):
    print("  ", name)

## 5. Enter the structure you are looking for

In [ ]:
#@title Query structure
QUERY_SMILES = "CC(=O)Oc1ccccc1C(=O)O"  #@param {type:"string"}

from rdkit import Chem
from rdkit.Chem import Draw

molecule = Chem.MolFromSmiles(QUERY_SMILES)
assert molecule is not None, "That SMILES could not be parsed — check it."
Chem.RemoveStereochemistry(molecule)
print("Canonical (no stereo):", Chem.MolToSmiles(molecule))
print("InChIKey (no stereo): ", Chem.MolToInchiKey(molecule))
Draw.MolToImage(molecule, size=(350, 350))

## 6. Search

The first pass over a document is the slow one — every page is segmented,
classified and read. That work is query-independent and cached on Drive, so
section 9 (a different molecule, same documents) returns immediately.

In [ ]:
#@title Run the search
import os
from structure_finder import find_structure, format_summary

#@markdown Folder to search (the upload folder by default; a Drive path also works):
DOCUMENTS_DIR = "/content/documents"  #@param {type:"string"}
#@markdown Recognition engine — `decimer` unless you are on a Python 3.11 runtime:
RECOGNIZER = "decimer"  #@param ["decimer", "molgrapher", "ensemble"]
#@markdown Page rendering resolution. 300 matches DECIMER-Segmentation's own
#@markdown default and is the sweet spot; 400 can help tiny depictions, but 600+
#@markdown is slow and memory-hungry for little gain.
DPI = 300  #@param [200, 300, 400] {type:"raw"}

assert os.path.isdir(DOCUMENTS_DIR), f"{DOCUMENTS_DIR} is not a directory"

results = find_structure(
    documents=[DOCUMENTS_DIR],
    smiles=QUERY_SMILES,
    use_drive=True,
    segmenter="decimer",                                  # DECIMER-Segmentation
    classifier="molclassifier" if os.path.isdir("/content/MolClassifier") else "none",
    recognizer=RECOGNIZER,
    match_modes=("exact", "connectivity", "tautomer"),
    dpi=DPI,
)

print(format_summary(results))

## 7. See the matches in context

In [ ]:
#@title Write the report and show the annotated pages
from pathlib import Path
from IPython.display import Image, display
from structure_finder import save_all

output_dir = Path(workspace.outputs) / "latest"
written = save_all(results, output_dir, annotate=True)
print("Report:", output_dir, "\n")

pages = written.get("annotated_pages", [])
for page_path in pages:
    print(page_path)
    display(Image(filename=page_path, width=760))

if not pages:
    print("No image hits to annotate — see the summary above, and section 10.")

In [ ]:
#@title Hits as a table (and download the CSV)
import pandas as pd

hits = pd.DataFrame(results["hits"])
display(hits if len(hits) else "No matches.")

if len(hits):
    from google.colab import files
    hits.to_csv("/content/hits.csv", index=False)
    files.download("/content/hits.csv")

## 8. Everything the pipeline read

`structure_search_extractions.jsonl` lists **every** chemical image found — page,
bounding box, class and SMILES — whether or not it matched. This is what you
check when a search comes back empty: it tells you whether the depiction was
missed by the segmenter or misread by the recognizer.

In [ ]:
#@title Browse all recognised structures
import json, pandas as pd

rows = []
for line in open(output_dir / "structure_search_extractions.jsonl"):
    record = json.loads(line)
    for figure in record["figures"]:
        rows.append({
            "document": record["document"]["filename"],
            "page": figure["page"],
            "class": figure["figure_class"],
            "smiles": figure["canonical_smiles"],
            "confidence": figure["recognition_confidence"],
        })

everything = pd.DataFrame(rows)
print(f"{len(everything)} chemical image(s) found")
display(everything.head(50))

## 9. Search another structure — fast, the cache is already warm

In [ ]:
#@title Another query over the same documents
SECOND_QUERY = "Cn1cnc2c1c(=O)n(C)c(=O)n2C"  #@param {type:"string"}

results_2 = find_structure(
    documents=[DOCUMENTS_DIR],
    smiles=SECOND_QUERY,
    use_drive=True,
    recognizer=RECOGNIZER,
    match_modes=("exact", "connectivity", "tautomer"),
    dpi=DPI,
)
print(format_summary(results_2))

## 10. Reading the result honestly

From the PatCID paper (Table 3), the full segment → classify → recognise chain
scores **54.5% precision / 46.0% recall** on its random benchmark and
**41.3% / 44.5%** on the deliberately hard one (older documents, non-US patent
offices, less standard drawing styles).

So: **a hit is strong evidence, a miss is weak evidence.**

If you expected a hit and got none, widen the search before concluding anything:

```python
results = find_structure(
    documents=[DOCUMENTS_DIR],
    smiles=QUERY_SMILES,
    use_drive=True,
    recognizer="ensemble",          # DECIMER + MolGrapher, keeps the better read
    match_modes=("exact", "connectivity", "tautomer", "similarity"),
    similarity_threshold=0.85,      # catches small recognition errors
    dpi=400,                        # small or low-quality depictions
)
```

then look at section 8 to see what the pipeline actually read.

**Markush structures** (generic structures with R groups) have no single molecule
behind them, so they can never match a concrete SMILES. They are counted
separately in the summary. Search their scaffold instead:

```python
find_structure(documents=[DOCUMENTS_DIR], smiles="smarts:c1ccc2[nH]ccc2c1",
               match_modes=("substructure",), use_drive=True)
```

**Word documents** keep real page numbers only because LibreOffice is installed
in section 2. Without it, the tool falls back to the images embedded in the
`.docx` and numbers them sequentially.